# Stage 3 CLIP / Hybrid Coarse Benchmark Clean (Kaggle)

Coarse-only benchmark for the hybrid direction. The notebook saves a compact result archive and an error traceback if something fails.


In [ ]:
import json
import shutil
import subprocess
import traceback
from pathlib import Path
from collections import Counter

RUN_NAME = "stage3_clip_hybrid_coarse_clean"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_clip_hybrid_coarse_clean.tar.gz")
LOG_PATH = WORK_DIR / "run_log.txt"
ERROR_PATH = WORK_DIR / "error_traceback.txt"

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
TEXT_PROMPTS = {
    "insulator_ok": "a clear photo crop of an intact power line insulator with regular shape and no visible damage",
    "defect_flashover": "a photo crop of a power line insulator with flashover burn marks dark surface traces or localized surface damage",
    "defect_broken": "a photo crop of a broken power line insulator with missing fragment damaged edge or structural break",
}

def log(msg):
    print(msg, flush=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

def sh(args, cwd: Path | None = None, check: bool = True):
    log("$ " + " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        log(p.stdout)
    if p.stderr:
        log(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    candidates.extend([
        split_root / p,
        split_root.parent / p,
        split_root.parent / "crops" / row.get("split", "") / row.get("coarse_class", "") / p.name,
    ])
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f"crop not found for {row.get('record_id')}: {row.get('crop_path')}")

def package_outputs():
    try:
        if ARCHIVE_PATH.exists():
            ARCHIVE_PATH.unlink()
        sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(WORK_DIR.parent), RUN_NAME], check=False)
        log(f"Archive: {ARCHIVE_PATH}")
    except Exception:
        print(traceback.format_exc(), flush=True)

try:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    log(f"RUN_NAME: {RUN_NAME}")
    log(f"WORK_DIR: {WORK_DIR}")

    gpu = sh(["nvidia-smi"], check=False)
    if gpu.returncode != 0:
        log("GPU not available; continuing on CPU will be slower.")
    else:
        log("GPU available")

    DATA_ROOT = None
    for root in DATASET_ROOT_CANDIDATES:
        if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
            DATA_ROOT = root
            break
    if DATA_ROOT is None:
        raise FileNotFoundError("Could not find stage3_regrouped_v2 train/val JSONL in Kaggle inputs")

    train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
    val_jsonl = DATA_ROOT / VAL_JSONL_REL
    train_root = train_jsonl.parent
    val_root = val_jsonl.parent
    log(f"DATA_ROOT: {DATA_ROOT}")
    log(f"train_jsonl: {train_jsonl}")
    log(f"val_jsonl: {val_jsonl}")

    sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "scikit-learn", "tabulate"])

    import numpy as np
    import pandas as pd
    import torch
    from PIL import Image
    from IPython.display import display
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    from sklearn.preprocessing import normalize
    from transformers import CLIPModel, CLIPProcessor

    train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
    val_rows = [r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
    log(f"train rows: {len(train_rows)} {Counter(r['coarse_class'] for r in train_rows)}")
    log(f"val rows: {len(val_rows)} {Counter(r['coarse_class'] for r in val_rows)}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"device: {device}")
    model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    model.eval()

    def embed_images(rows, split_root: Path, batch_size: int = 32):
        feats=[]
        labels=[]
        ids=[]
        with torch.no_grad():
            for start in range(0, len(rows), batch_size):
                batch = rows[start:start+batch_size]
                images=[]
                for row in batch:
                    img = Image.open(resolve_crop(row, split_root)).convert("RGB")
                    images.append(img)
                inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
                emb = model.get_image_features(**inputs)
                feats.append(emb.detach().cpu().numpy())
                labels.extend([r["coarse_class"] for r in batch])
                ids.extend([r["record_id"] for r in batch])
        return normalize(np.vstack(feats)), np.array(labels), ids

    def embed_texts():
        texts=[TEXT_PROMPTS[label] for label in LABELS]
        with torch.no_grad():
            inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
            emb = model.get_text_features(**inputs).detach().cpu().numpy()
        return normalize(emb)

    X_train, y_train, train_ids = embed_images(train_rows, train_root)
    X_val, y_val, val_ids = embed_images(val_rows, val_root)
    T = embed_texts()
    log(f"embeddings: {X_train.shape} {X_val.shape} {T.shape}")

    sim = X_val @ T.T
    zs_pred = np.array([LABELS[i] for i in sim.argmax(axis=1)])

    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
    clf.fit(X_train, y_train)
    lp_pred = clf.predict(X_val)

    proba = clf.predict_proba(X_val)
    maxp = proba.max(axis=1)
    hybrid_pred = np.array([p if conf >= 0.45 else "unknown" for p, conf in zip(lp_pred, maxp)])

    def macro_f1_with_unknown(y_true, y_pred):
        labels = ["insulator_ok", "defect_flashover", "defect_broken", "unknown", "other"]
        return f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    def summarize(name, pred):
        return {
            "method": name,
            "accuracy": accuracy_score(y_val, pred),
            "macro_f1_3class": f1_score(y_val, pred, labels=LABELS, average="macro", zero_division=0),
            "macro_f1_with_unknown": macro_f1_with_unknown(y_val, pred),
            "ok_recall": ((pred == "insulator_ok") & (y_val == "insulator_ok")).sum() / max((y_val == "insulator_ok").sum(), 1),
            "flashover_recall": ((pred == "defect_flashover") & (y_val == "defect_flashover")).sum() / max((y_val == "defect_flashover").sum(), 1),
            "broken_recall": ((pred == "defect_broken") & (y_val == "defect_broken")).sum() / max((y_val == "defect_broken").sum(), 1),
        }

    summary = pd.DataFrame([
        summarize("clip_zero_shot", zs_pred),
        summarize("clip_linear_probe", lp_pred),
        summarize("clip_linear_probe_unknown_threshold_045", hybrid_pred),
    ])
    display(summary)
    summary.to_csv(WORK_DIR / "coarse_benchmark_summary.csv", index=False)

    report_lines = []
    for name, pred in [("clip_zero_shot", zs_pred), ("clip_linear_probe", lp_pred), ("clip_linear_probe_unknown_threshold_045", hybrid_pred)]:
        report_lines.append("\n" + name)
        report_lines.append(classification_report(y_val, pred, labels=["insulator_ok", "defect_flashover", "defect_broken", "unknown"], zero_division=0))
    (WORK_DIR / "classification_reports.txt").write_text("\n".join(report_lines), encoding="utf-8")

    pred_rows=[]
    for rid, gt, zs, lp, hp, conf in zip(val_ids, y_val, zs_pred, lp_pred, hybrid_pred, maxp):
        pred_rows.append({
            "record_id": rid,
            "gt": gt,
            "clip_zero_shot": zs,
            "clip_linear_probe": lp,
            "clip_linear_probe_confidence": float(conf),
            "clip_linear_probe_unknown_threshold_045": hp,
        })
    pd.DataFrame(pred_rows).to_csv(WORK_DIR / "coarse_predictions.csv", index=False)

    lines = [
        "# Stage 3 CLIP / Hybrid Coarse Benchmark Clean",
        "",
        f"CLIP model: `{CLIP_MODEL_ID}`",
        "",
        summary.to_markdown(index=False),
        "",
        "TL-CLIP status: not executed because no public runnable weights were attached to this Kaggle run.",
    ]
    (WORK_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
    log(summary.to_string(index=False))
    package_outputs()
except Exception:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    tb = traceback.format_exc()
    ERROR_PATH.write_text(tb, encoding="utf-8")
    print(tb, flush=True)
    package_outputs()
    raise
